# ME 3300 Prelab 03 Walkthrough — Automation & Derivatives

Lab 03 brings your first **DAQ automation** (scripting the ADS with `dwfpy`)
and your first **numerical derivative**. This walkthrough teaches the new
Python patterns on simulated signals — no hardware needed. Real acquisition
happens in lab.

Run cell by cell; report **CHECKPOINT** values in the Prelab 03 quiz on
Canvas. Select your `.venv` kernel first.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)   # seeded: everyone gets the same numbers
print('imports OK')

## 1. The `with` block — open, use, always close

Talking to hardware means *claiming* it: while your script holds the ADS,
nothing else (including WaveForms) can use it. So acquisition code must
guarantee the device is released — **even if your code crashes halfway**.
Python's *context manager* (`with ... as ...:`) does exactly that: it opens
a resource at the top of the block and *always* closes it when the block
exits, error or no error. Here it is explicitly, with a file standing in
for the ADS:

In [ ]:
# A file is a stand-in for the ADS: a resource that must be opened and closed
with open('demo.txt', 'w') as f:
    f.write('hello ME 3300')
    print('inside the block: file open?', not f.closed)

print('after the block:  file open?', not f.closed)

In lab, the same pattern claims and releases the ADS:

```python
with dwf.Device() as device:      # opened here...
    ...                           # use the instruments
# ...and automatically closed here, even if an error occurred
```

That auto-close matters: a crashed script that left the device claimed
would lock you out until you unplugged the USB cable.

**CHECKPOINT 1:** What does the *last* line of the cell above print —
`True` or `False`? Why is that the answer, in one sentence?

## 2. `input()` — pausing for the human

`input('message')` prints the message and pauses until you press Enter
(returning whatever you typed). Acquisition scripts use it to wait while you
physically position the apparatus. Try it — run this cell, type your name in
the box that appears (top of the VS Code window), press Enter:

In [ ]:
name = input('Type your name and press Enter: ')
print(f'Hello, {name}! The script waited for you.')

## 3. Lists, `.append`, and `zip`

In Lab 02 you pre-allocated arrays with `np.zeros(...)` and filled slots by
index. When results simply arrive one at a time, a plain **list** you
`.append` to is more natural:

In [ ]:
measurements = []            # empty list
for trial in range(4):
    fake_reading = 1.65 + 0.1*trial
    measurements.append(fake_reading)

print(measurements)
measurements = np.array(measurements)   # convert for math
print('mean:', measurements.mean())

In [ ]:
# zip: loop over two lists IN PARALLEL, one matched pair per pass
positions   = ['right', 'up', 'left', 'down']
known_accel = [0.0, 9.81, 0.0, -9.81]

for pos, accel in zip(positions, known_accel):
    print(f"at '{pos}' the sensor should read {accel:+.2f} m/s^2")

Compare with `enumerate` (Lab 02): `enumerate` pairs each element with
its *index*; `zip` pairs elements of one list with elements of *another*.

**CHECKPOINT 2:** Using `zip`, loop over `[1, 2, 3]` and `[10, 20, 30]` and
append the *products* to a list. What is the sum of that list?

## 4. `np.column_stack` — saving multi-column results

In [ ]:
t = np.arange(0, 1, 0.25)
x = t**2
table = np.column_stack([t, x])      # two columns, side by side
print(table)
np.savetxt('practice_table.csv', table, header='t_s,x_m', delimiter=',')

# ...and np.loadtxt reads it back, skipping the '#' header automatically:
back = np.loadtxt('practice_table.csv', delimiter=',', comments='#')
print('round trip OK:', np.allclose(table, back))

Note what `np.savetxt` did with `header=`: it wrote the line prefixed
with `#`. That's why `comments='#'` on the way back in skips it — the same
convention your saved calibration files use.

## 5. Finding events: `np.diff` and `np.argmax`

In Lab 02 you trimmed your swing recording *interactively*: hover the plotly
figure, read the release index, slice. That works great for one file. When an
analysis must process *many* recordings, it pays to automate the detection —
Lab 03's optional postlab exploration builds exactly such a detector, and this
is its core trick.

A held-then-released signal barely changes while held, then changes fast.
`np.diff` computes the change between adjacent samples:

In [ ]:
# Fake "held then released" signal
t_sig = np.arange(0, 4, 0.01)
sig   = np.where(t_sig < 1.5, 2.0, 2.0 + 0.8*np.sin(6*(t_sig-1.5)))
sig  += rng.normal(0, 0.001, t_sig.size)

d = np.abs(np.diff(sig))
print(f"typical change while held:   {d[:100].mean():.5f}")
print(f"biggest change overall:      {d.max():.5f}")

In [ ]:
# The comparison d > 0.01 gives an array of True/False.
# np.argmax returns the index of the FIRST True (True counts as 1).
idx0 = np.argmax(d > 0.01)
print(f"release detected at index {idx0}, t = {t_sig[idx0]:.2f} s")

# Trim: keep everything from idx0 on, and restart the clock
t_trim   = t_sig[idx0:] - t_sig[idx0]
sig_trim = sig[idx0:]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6.5, 5), sharey=True)
ax1.plot(t_sig, sig); ax1.set_title('raw'); ax1.grid(True)
ax2.plot(t_trim, sig_trim); ax2.set_title('trimmed (t=0 at release)'); ax2.grid(True)
ax2.set_xlabel('Time (s)')
plt.tight_layout()
plt.show()

Compare the detector's index against what you'd read off an interactive plot —
same answer, no hovering. The catch: the `0.01` threshold was *designed*
against this signal's noise level. Choose it badly (or change the sample
rate!) and the detector misfires — the manual explains why, and shows a more
robust condition. If a threshold ever misfires on your real data, *plot* `d`
and look at it — choosing thresholds by eye is normal engineering practice.

## 6. `np.gradient` — the derivative, done right

`np.diff` (previous section) gives differences between neighbors: N points in,
**N−1** out, each value landing *between* two samples. For a derivative you
plot against the original time axis, that off-by-one-and-a-half-shift is a
nuisance. **`np.gradient`** uses *central* differences (each slope computed
from both neighbors) and returns the **same length** as its input:

In [ ]:
dt = 0.01
t_sig = np.arange(0, 2, dt)
x_sig = np.sin(2*np.pi*t_sig)               # position
v_true = 2*np.pi*np.cos(2*np.pi*t_sig)      # exact derivative

v_num = np.gradient(x_sig, dt)              # numerical derivative

print(f"len(x) = {len(x_sig)}, len(gradient) = {len(v_num)}, "
      f"len(diff) = {len(np.diff(x_sig))}")
print(f"max error vs exact: {np.abs(v_num - v_true).max():.4f}")

**CHECKPOINT 3:** What is `len(np.gradient(x))` for an array `x` of
length 200? What is `len(np.diff(x))`?

### Why derivatives amplify noise

Now the important physics-of-measurement lesson. Add a little noise to the
position signal and differentiate again:

In [ ]:
x_noisy = x_sig + rng.normal(0, 0.001, x_sig.size)   # tiny 0.001 noise!
v_noisy = np.gradient(x_noisy, dt)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6.5, 5), sharex=True)
ax1.plot(t_sig, x_noisy, 'b-', linewidth=1)
ax1.set_ylabel('position'); ax1.set_title('signal: noise is invisible')
ax1.grid(True)
ax2.plot(t_sig, v_noisy, 'b-', linewidth=0.8, label='gradient of noisy signal')
ax2.plot(t_sig, v_true, 'r-', linewidth=2, label='true derivative')
ax2.set_ylabel('velocity'); ax2.set_xlabel('time (s)')
ax2.set_title('derivative: the same noise, amplified')
ax2.legend(); ax2.grid(True)
plt.tight_layout(); plt.show()

noise_amplification = v_noisy.std() / v_true.std()
print(f"derivative noise is clearly visible even though position noise "
      f"was only 0.001")

The noise was invisible in the position trace but dominates the
derivative. Differentiation divides each sample-to-sample noise wiggle by
the tiny `dt`, magnifying it enormously. **This is why your calculated
normal acceleration in lab will be much noisier than the accelerometer's
direct measurement** — and it's why we still record fast (the dense samples
let the derivative trace the real curve; the noise cost is unavoidable
either way).

**CHECKPOINT 4:** Re-run the cell above with noise `0.005` instead of
`0.001`. Roughly what is the peak spurious velocity (largest wrong value)
in the derivative plot? Report to the nearest whole number.

## 7. Preview: the dwfpy pattern

You can't run this at home without an ADS, but read it now so it's familiar
in lab (do NOT run this cell unless hardware is attached):

```python
import dwfpy as dwf

with dwf.Device() as device:            # auto-closes, even on errors
    scope = device.analog_input         # the Scope, as an object
    scope['ch1'].setup(range=5.0)       # GUI Channel 1, +/-5 V range
    scope.single(sample_rate=1000,      # = typing 1000 Hz in the GUI
                 buffer_size=2000,
                 configure=True, start=True)   # starts, then waits for Done
    volts = scope['ch1'].get_data()     # the samples, as a NumPy array
```

Every line maps to a GUI action you performed in Lab 02. Two handy details:
dwfpy channel labels match the GUI (`'ch1'` is GUI Channel 1, `'ch2'` is
Channel 2), and with `start=True` the `single(...)` call starts the
acquisition and *waits* until the hardware reports Done before returning —
that is why the cell pauses for the acquisition time.

**CHECKPOINT 5:** In the code above, which GUI channel does
`scope['ch1'].get_data()` read from? What one change reads GUI Channel 2?

## Done!

| Skill | Where you'll use it in lab |
|---|---|
| `with` context manager | claiming and releasing the ADS safely |
| `input()` | pausing between the 4 calibration orientations |
| list + `.append`, `zip` | collecting the 4 calibration means |
| `np.column_stack` / `loadtxt(comments='#')` | saving/loading calibrations |
| `np.diff` + `np.argmax` | (optional) automating the release detection |
| `np.gradient` | angular velocity from the angle signal |
| noise-amplification insight | explaining your comparison plot |

Report the checkpoints in the **Prelab 03 quiz on Canvas**.